In [1]:
import pandas as pd

# Đọc các file pickle từ thư mục ../data/
train_df = pd.read_pickle('../data/train_fe.pkl')
val_df = pd.read_pickle('../data/val_fe.pkl')
test_df = pd.read_pickle('../data/test_fe.pkl')

# Kiểm tra dữ liệu (ví dụ: in ra số dòng và cột)
print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")


Train shape: (1577139, 71)
Val shape: (337206, 71)
Test shape: (373699, 71)


In [2]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression, SelectPercentile
from xgboost import XGBRegressor, XGBClassifier
import pandas as pd
import numpy as np
import json

# ════════════════════════════════════════════════════════════════
# 1. CHUẨN BỊ DỮ LIỆU
# ════════════════════════════════════════════════════════════════
DROP_COLS = ['fullVisitorId', 'target_revenue', 'target_log_revenue', 'Fold', 'Split_Type']

y_train = train_df['target_log_revenue']
X_train = train_df.drop(columns=DROP_COLS)

y_val = val_df['target_log_revenue']
X_val = val_df.drop(columns=DROP_COLS)

print(f"Số lượng đặc trưng ban đầu: {len(X_train.columns)}")

# ════════════════════════════════════════════════════════════════
# LỚP 1: VARIANCE THRESHOLD
# ════════════════════════════════════════════════════════════════
var_selector = VarianceThreshold(threshold=(.995 * (1 - .995)))
var_selector.fit(X_train)

X_train_v = X_train.loc[:, var_selector.get_support()]
X_val_v   = X_val.loc[:, var_selector.get_support()]
print(f"1. Sau Variance Threshold: Còn {X_train_v.shape[1]} biến")

# ════════════════════════════════════════════════════════════════
# LỚP 2: CORRELATION FILTER
# ════════════════════════════════════════════════════════════════
corr_matrix = X_train_v.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

X_train_c = X_train_v.drop(columns=to_drop_corr)
X_val_c   = X_val_v.drop(columns=to_drop_corr)
print(f"2. Sau Correlation Filter: Còn {X_train_c.shape[1]} biến")

# ════════════════════════════════════════════════════════════════
# LỚP 3: MUTUAL INFORMATION
# ════════════════════════════════════════════════════════════════
print("3. Đang thực hiện Mutual Information...")
mi_percentile = 95 if X_train_c.shape[1] < 100 else 90
mi_selector = SelectPercentile(mutual_info_regression, percentile=mi_percentile)

# fillna(-1) chỉ dùng trong MI (không ảnh hưởng X_train_c gốc)
mi_selector.fit(X_train_c.fillna(-1), y_train)  

X_train_mi = X_train_c.loc[:, mi_selector.get_support()]
X_val_mi   = X_val_c.loc[:, mi_selector.get_support()]
print(f"3. Sau Mutual Information (Top {mi_percentile}%): Còn {X_train_mi.shape[1]} biến")

# ════════════════════════════════════════════════════════════════
# LỚP 4: 2-STAGE XGBOOST FEATURE IMPORTANCE (Union Approach)
# ════════════════════════════════════════════════════════════════
print("4. Đang chạy XGBoost Importance cho cả 2 tầng (Classification + Regression)...")

# --- 4A. CLASSIFICATION IMPORTANCE (Ai sẽ mua?) ---
y_train_buy = (y_train > 0).astype(int)
y_val_buy   = (y_val > 0).astype(int)

model_fs_clf = XGBClassifier(
    n_estimators=500, max_depth=3, learning_rate=0.05, 
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1,
    early_stopping_rounds=30, eval_metric='logloss'
)

model_fs_clf.fit(
    X_train_mi, y_train_buy,
    eval_set=[(X_val_mi, y_val_buy)],
    verbose=False
)

imp_clf = pd.DataFrame({
    'feature': X_train_mi.columns,
    'importance': model_fs_clf.feature_importances_
}).sort_values(by='importance', ascending=False)

# Cumulative 95% cho Classification
cutoff_clf = (imp_clf['importance'].cumsum() <= 0.95).sum()
keep_clf = imp_clf.iloc[:cutoff_clf + 1]['feature'].tolist()

# --- 4B. REGRESSION IMPORTANCE (Mua bao nhiêu? - Chỉ dùng tập Buyers) ---
mask_buyers_train = y_train > 0
mask_buyers_val   = y_val > 0

model_fs_reg = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.05, 
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1,
    early_stopping_rounds=30, eval_metric='rmse'
)

model_fs_reg.fit(
    X_train_mi[mask_buyers_train], y_train[mask_buyers_train],
    eval_set=[(X_val_mi[mask_buyers_val], y_val[mask_buyers_val])],
    verbose=False
)

imp_reg = pd.DataFrame({
    'feature': X_train_mi.columns,
    'importance': model_fs_reg.feature_importances_
}).sort_values(by='importance', ascending=False)

# Cumulative 95% cho Regression
cutoff_reg = (imp_reg['importance'].cumsum() <= 0.95).sum()
keep_reg = imp_reg.iloc[:cutoff_reg + 1]['feature'].tolist()

# --- 4C. GỘP ĐẶC TRƯNG TỪ 2 TẦNG ---
final_features = list(set(keep_clf) | set(keep_reg))

print(f"   + Giữ lại từ Classifier: {len(keep_clf)} biến")
print(f"   + Giữ lại từ Regressor:  {len(keep_reg)} biến")
print(f"4. Sau Model Importance (Lấy Hợp - Union): Còn {len(final_features)} biến")


metadata_train_val = ['fullVisitorId', 'target_revenue', 'target_log_revenue', 'Fold', 'Split_Type']
metadata_test      = ['fullVisitorId', 'Fold', 'Split_Type']

train_df_final = train_df[metadata_train_val + final_features]
val_df_final   = val_df[metadata_train_val + final_features]
test_df_final  = test_df[metadata_test + final_features]

# Lưu pickle
train_df_final.to_pickle('../data/train_final.pkl')
val_df_final.to_pickle('../data/val_final.pkl')
test_df_final.to_pickle('../data/test_final.pkl')

features_dict = {
    'clf_features': keep_clf,
    'reg_features': keep_reg,
    'union_features': final_features
}
with open('../data/final_features.json', 'w') as f:
    json.dump(features_dict, f, indent=2)


print(f"\n=> HOÀN TẤT: Còn lại {len(final_features)} đặc trưng tốt nhất cho mô hình 2 tầng.")
print("Dữ liệu cuối cùng đã được lưu thành công.")

print("\n--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (CLASSIFICATION) ---")
print(imp_clf.head(10))

print("\n--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (REGRESSION) ---")
print(imp_reg.head(10))


Số lượng đặc trưng ban đầu: 66
1. Sau Variance Threshold: Còn 62 biến
2. Sau Correlation Filter: Còn 44 biến
3. Đang thực hiện Mutual Information...
3. Sau Mutual Information (Top 95%): Còn 41 biến
4. Đang chạy XGBoost Importance cho cả 2 tầng (Classification + Regression)...
   + Giữ lại từ Classifier: 30 biến
   + Giữ lại từ Regressor:  30 biến
4. Sau Model Importance (Lấy Hợp - Union): Còn 40 biến

=> HOÀN TẤT: Còn lại 40 đặc trưng tinh túy nhất cho mô hình 2 tầng.
Dữ liệu cuối cùng đã được lưu thành công.

--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (CLASSIFICATION) ---
                             feature  importance
33                  hibernation_flag    0.183802
32               days_since_purchase    0.115399
17  geoNetwork_subContinent_enc_mean    0.091096
15       geoNetwork_country_enc_mean    0.085225
37           interaction_hits_visits    0.084897
9      totals_transactionRevenue_sum    0.047045
30                      recency_days    0.046193
18          channelGrouping_enc_me

Stage 1: Classification

In [3]:
from xgboost import XGBClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score, classification_report,
    confusion_matrix, fbeta_score, brier_score_loss
)
import numpy as np
import pandas as pd
import json

# ════════════════════════════════════════════════════════════
# LOAD DATA & FEATURES
# ════════════════════════════════════════════════════════════
train_df = pd.read_pickle('../data/train_final.pkl')
val_df   = pd.read_pickle('../data/val_final.pkl')

with open('../data/final_features.json') as f:
    features_dict = json.load(f)

clf_features = features_dict['clf_features']
X_train = train_df[clf_features]
X_val   = val_df[clf_features]

y_train_buy = (train_df['target_revenue'] > 0).astype(int)
y_val_buy   = (val_df['target_revenue'] > 0).astype(int)

neg = (y_train_buy == 0).sum()
pos = (y_train_buy == 1).sum()

# sqrt cho cân bằng PR-AUC vs Recall, tốt hơn full ratio theo thực nghiệm
spw = np.sqrt(neg / pos)
print(f"Negative: {neg:,} | Positive: {pos:,} | SPW: {spw:.2f}x")

# ════════════════════════════════════════════════════════════
# TEMPORAL CALIBRATION SPLIT (không shuffle — tránh temporal leakage)
# cal_train_folds: train classifier
# cal_val_folds:   fit Isotonic (truly out-of-fold)
# ════════════════════════════════════════════════════════════
train_folds     = sorted(train_df['Fold'].unique())
n_cal_train     = int(len(train_folds) * 0.8)
cal_train_folds = train_folds[:n_cal_train]
cal_val_folds   = train_folds[n_cal_train:]

cal_train_mask = train_df['Fold'].isin(cal_train_folds)
cal_val_mask   = train_df['Fold'].isin(cal_val_folds)

X_cal_train = X_train[cal_train_mask]
y_cal_train = y_train_buy[cal_train_mask]
X_cal_val   = X_train[cal_val_mask]
y_cal_val   = y_train_buy[cal_val_mask]

print(f"Cal train folds: {cal_train_folds} | Cal val folds: {cal_val_folds}")

# Kiểm tra đủ buyers trong cal_val để isotonic ổn định
n_pos_cal = y_cal_val.sum()
print(f"Cal val buyers: {n_pos_cal:,} / {len(y_cal_val):,} total")
if n_pos_cal < 50:
    print("⚠️  Quá ít buyers trong cal_val — isotonic có thể không ổn định")

# ════════════════════════════════════════════════════════════
# BASE XGB PARAMS (dùng chung cho cả 2 lần fit)
# ════════════════════════════════════════════════════════════
BASE_PARAMS = dict(
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='aucpr',
    n_jobs=-1,
)

# ════════════════════════════════════════════════════════════
# ENSEMBLE + ISOTONIC CALIBRATION
#
# Mỗi seed:
#   1. Fit clf trên cal_train_folds → early stop trên cal_val_folds
#   2. Fit Isotonic trên p_oof vs y_cal_val
#   3. Retrain clf_full trên TOÀN BỘ train_df với best_iteration
#   4. Apply isotonic lên clf_full predictions
# ════════════════════════════════════════════════════════════
N_RUNS      = 5
p_cal_sum   = np.zeros(len(val_df))
pr_auc_list = []
brier_list  = []

print(f"\nTraining {N_RUNS} classifiers...")

for seed in range(N_RUNS):
    # ── Bước 1: Fit trên cal_train để tìm best_iteration ──
    clf_cal = XGBClassifier(
        **BASE_PARAMS,
        n_estimators=1000,
        early_stopping_rounds=30,
        random_state=seed,
    )
    clf_cal.fit(
        X_cal_train, y_cal_train,
        eval_set=[(X_cal_val, y_cal_val)],
        verbose=False,
    )
    best_iter = clf_cal.best_iteration

    # ── Bước 2: Fit Isotonic trên truly OOF predictions ──
    p_oof = clf_cal.predict_proba(X_cal_val)[:, 1]
    iso   = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_oof, y_cal_val)

    # ── Bước 3: Retrain full với best_iteration
    clf_full = XGBClassifier(
        **BASE_PARAMS,
        n_estimators=best_iter,
        early_stopping_rounds=None,   
        random_state=seed,
    )
    clf_full.fit(X_train, y_train_buy, verbose=False)

    # ── Bước 4: Predict + calibrate ──
    p_raw = clf_full.predict_proba(X_val)[:, 1]
    p_cal = iso.predict(p_raw)
    p_cal_sum += p_cal

    pr = average_precision_score(y_val_buy, p_cal)
    b  = brier_score_loss(y_val_buy, p_cal)
    pr_auc_list.append(pr)
    brier_list.append(b)
    print(f"  seed={seed} | PR-AUC={pr:.4f} | Brier={b:.6f} | iter={best_iter}")

p_avg           = p_cal_sum / N_RUNS
val_df['p_buy'] = p_avg

ens_pr  = average_precision_score(y_val_buy, p_avg)
ens_b   = brier_score_loss(y_val_buy, p_avg)
true_rate = y_val_buy.mean()

print(f"\nEnsemble PR-AUC: {ens_pr:.4f}")
print(f"Ensemble Brier:  {ens_b:.6f}")
print(f"Mean p_buy:      {p_avg.mean():.6f} (true rate: {true_rate:.6f})")

# Cảnh báo nếu calibration lệch nhiều so với true rate
calib_ratio = p_avg.mean() / true_rate
if not (0.8 <= calib_ratio <= 1.2):
    print(f"⚠️  Calibration lệch {calib_ratio:.2f}x so với true rate")

# ════════════════════════════════════════════════════════════
# F2 THRESHOLD — Recall quan trọng hơn Precision
# Bỏ sót buyer ở Stage 1 → Stage 2 mất signal
# ════════════════════════════════════════════════════════════
thresholds = np.arange(0.001, 0.5, 0.001)
f2_scores  = [
    fbeta_score(y_val_buy, (p_avg >= t).astype(int), beta=2, zero_division=0)
    for t in thresholds
]

best_t  = thresholds[np.argmax(f2_scores)]
best_f2 = max(f2_scores)

print(f"\nF2 Threshold: {best_t:.4f} | F2={best_f2:.4f}")

y_pred = (p_avg >= best_t).astype(int)
print(classification_report(y_val_buy, y_pred, digits=4))

cm = confusion_matrix(y_val_buy, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"TP: {tp:,} ({tp / (tp + fn) * 100:.1f}% buyers found)")
print(f"FN: {fn:,} ({fn / (tp + fn) * 100:.1f}% buyers missed)")

print(f"\n✅ Saved: val_df['p_buy'] | best_t={best_t:.4f}")

Negative: 1,575,674 | Positive: 1,465 | SPW: 32.80x
Cal train folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | Cal val folds: [np.int64(5)]
Cal val buyers: 229 / 292,191 total

Training 5 classifiers...
  seed=0 | PR-AUC=0.0661 | Brier=0.000716 | iter=99
  seed=1 | PR-AUC=0.0679 | Brier=0.000720 | iter=61
  seed=2 | PR-AUC=0.0656 | Brier=0.000713 | iter=60
  seed=3 | PR-AUC=0.0712 | Brier=0.000720 | iter=74
  seed=4 | PR-AUC=0.0640 | Brier=0.000718 | iter=76

Ensemble PR-AUC: 0.0750
Ensemble Brier:  0.000715
Mean p_buy:      0.000730 (true rate: 0.000744)

F2 Threshold: 0.0460 | F2=0.2106
              precision    recall  f1-score   support

           0     0.9995    0.9976    0.9986    336955
           1     0.0906    0.3147    0.1407       251

    accuracy                         0.9971    337206
   macro avg     0.5450    0.6562    0.5696    337206
weighted avg     0.9988    0.9971    0.9979    337206

TP: 79 (31.5% buyers found)
FN: 172 (68.5% buyers missed)

✅ Sav

Stage 2: Regression

In [5]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

reg_features = features_dict['reg_features']
N_RUNS       = 5

# ── Chuẩn bị dữ liệu ──────────────────────────────────────
train_buyers = train_df[train_df['target_revenue'] > 0].copy()
val_buyers   = val_df[val_df['target_revenue'] > 0].copy()

X_train_b    = train_buyers[reg_features]
X_val_b      = val_buyers[reg_features]
y_train_b    = train_buyers['target_log_revenue']
y_val_b      = val_buyers['target_log_revenue']

p99_log      = y_train_b.quantile(0.99)
val_mask_buy = val_df['target_revenue'].values > 0

print(f"Train buyers: {len(train_buyers):,} | Val buyers: {len(val_buyers):,}")
print(f"p99_log: {p99_log:.4f}")

baseline_b = np.sqrt(mean_squared_error(y_val_b, np.zeros(len(y_val_b))))
print(f"Baseline (predict 0 on buyers): {baseline_b:.4f}\n")

# ── Ensemble với early stopping ────────────────────────────
e_sum  = np.zeros(len(val_df))
rmse_list = []

for seed in range(N_RUNS):
    reg = XGBRegressor(
        n_estimators=2000,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        eval_metric='rmse',
        reg_alpha=0.1,
        reg_lambda=1.0,
        early_stopping_rounds=50,   
        random_state=seed,
        n_jobs=-1,
    )
    reg.fit(
        X_train_b, y_train_b,
        eval_set=[(X_val_b, y_val_b)],
        verbose=False,
    )
    pred = reg.predict(val_df[reg_features]).clip(0, p99_log)
    e_sum += pred
    rmse = np.sqrt(mean_squared_error(y_val_b, pred[val_mask_buy]))
    rmse_list.append(rmse)
    print(f"  seed={seed} | RMSE={rmse:.4f} | iter={reg.best_iteration}")

E_monetary = e_sum / N_RUNS
print(f"\nEnsemble RMSE: {np.mean(rmse_list):.4f} ± {np.std(rmse_list):.4f}")
print(f"Baseline:      {baseline_b:.4f}")

Train buyers: 1,465 | Val buyers: 251
p99_log: 21.6405
Baseline (predict 0 on buyers): 18.0156

  seed=0 | RMSE=1.2028 | iter=93
  seed=1 | RMSE=1.1971 | iter=161
  seed=2 | RMSE=1.1945 | iter=295
  seed=3 | RMSE=1.2057 | iter=110
  seed=4 | RMSE=1.2105 | iter=84

Ensemble RMSE: 1.2021 ± 0.0058
Baseline:      18.0156


Combine and evaluate

In [6]:
# ════════════════════════════════════════════════════════════
# 4. COMBINE — P(Buy) × E(log_revenue | buyer)
# ════════════════════════════════════════════════════════════
P = val_df['p_buy'].values
final_log_pred = (P * E_monetary).clip(0)
val_df['final_log_pred'] = final_log_pred
 
# ════════════════════════════════════════════════════════════
# 5. EVALUATE COMBINE
# ════════════════════════════════════════════════════════════
log_true     = val_df['target_log_revenue'].values
y_true       = val_df['target_revenue'].values
buyers_mask  = y_true > 0
baseline_all = np.sqrt(mean_squared_error(log_true, np.zeros(len(log_true))))
rmse_final   = np.sqrt(mean_squared_error(log_true, final_log_pred))
beat         = '✅' if rmse_final < baseline_all else '❌'
 
print(f"\n{'='*55}")
print(f"COMBINE RESULTS")
print(f"{'='*55}")
print(f"Baseline (predict 0): {baseline_all:.4f}")
print(f"{beat} Final RMSE:          {rmse_final:.4f}")
 
top_k       = int(len(val_df) * 0.10)
top_idx     = np.argsort(final_log_pred)[::-1][:top_k]
rev_capture = y_true[top_idx].sum() / y_true.sum()
lift        = buyers_mask[top_idx].mean() / buyers_mask.mean()
 
print(f"\nRevenue Capture@10%: {rev_capture:.4f}")
print(f"Lift@10%:            {lift:.2f}x")
 
# ════════════════════════════════════════════════════════════
# 6. RANKING KHÁCH HÀNG THEO GIÁ TRỊ KỲ VỌNG
#    Sắp xếp toàn bộ val_df theo final_log_pred giảm dần
#    in ra top 20 với đầy đủ thông tin (Đổi sang USD)
# ════════════════════════════════════════════════════════════
ranking = (
    val_df[['p_buy', 'final_log_pred', 'target_revenue']]
    .copy()
    .assign(
        pred_revenue = np.expm1(val_df['final_log_pred']),  # chỉ để in
        is_buyer     = (val_df['target_revenue'] > 0).astype(int),
        percentile   = lambda df: (
            df['final_log_pred'].rank(pct=True) * 100
        ).round(1),
    )
    .sort_values('final_log_pred', ascending=False)
    .reset_index()                      # giữ fullVisitorId làm cột
    .rename(columns={'index': 'user_id'})
)
 
# Thêm rank
ranking.insert(0, 'rank', range(1, len(ranking) + 1))
 
print(f"\n{'='*85}")
print(f"BẢNG XẾP HẠNG KHÁCH HÀNG THEO GIÁ TRỊ KỲ VỌNG (ĐVT: USD)")
print(f"(top 20 / {len(ranking):,} users — log predict giảm dần)")
print(f"{'='*85}")
 
hdr = (f"{'Rank':>4} {'User ID':>20} {'Log Pred':>10} "
       f"{'P(Buy)':>8} {'Buyer':>6}")
print(hdr)
print("-" * len(hdr))
 
top20 = ranking.head(20)

for idx, row in top20.iterrows():
    buyer_mark = "✓" if row['is_buyer'] == 1 else ""
    print(f"{row['rank']:>4} "
          f"{row['user_id']:>20} "
          f"{row['final_log_pred']:>10.4f} "
          f"{row['p_buy']:>8.4f} "
          f"{buyer_mark:>5}")

 


COMBINE RESULTS
Baseline (predict 0): 0.4915
✅ Final RMSE:          0.4811

Revenue Capture@10%: 0.8917
Lift@10%:            9.04x

BẢNG XẾP HẠNG KHÁCH HÀNG THEO GIÁ TRỊ KỲ VỌNG (ĐVT: USD)
(top 20 / 337,206 users — log predict giảm dần)
Rank              User ID   Log Pred   P(Buy)  Buyer
----------------------------------------------------
 1.0            1643138.0    19.1769   0.8862     ✓
 2.0            1869041.0    12.9919   0.6333      
 3.0            1578473.0    10.6205   0.5719      
 4.0            1907548.0    10.5858   0.5369      
 5.0            1809032.0    10.3629   0.5196     ✓
 6.0            1826108.0    10.2494   0.5293      
 7.0            1578648.0     9.5759   0.4770      
 8.0            1779906.0     9.4894   0.5028     ✓
 9.0            1635095.0     9.3164   0.4846      
10.0            1714257.0     9.1886   0.4770      
11.0            1818960.0     9.1331   0.4770      
12.0            1863446.0     9.0277   0.4770      
13.0            1756016.0     8.